In [12]:
import json
from pathlib import Path

import polars as pl
import seaborn as sns
from pydantic import BaseModel
from tqdm import tqdm

from simple_evals.improvement.models.results import AllResults, RubricItem
from simple_evals.improvement.models.rag_logs import RAGLog
from simple_evals.improvement.paths import RESULTS_DIR
from simple_evals.improvement.readers import (
    EvalInputReader,
    get_alexandria_document_by_case_id,
    RagLogReader,
    ExampleLevelMetadataReader,
)

In [10]:
class Document(BaseModel):
    question: str
    criteria: str
    axis: str
    prompt_id: str
    atropos_case_id: str
    atropos_summary: str
    criterion_points: int


In [4]:
baseline_results_path = RESULTS_DIR / (
    "5df4ba309cb03369f6663786ae6a9904385524a9/"
    + "maverick/healthbench_llama-4-maverick_20251023_212754_allresults.json"
)
baseline_results = AllResults.from_file(baseline_results_path)

In [5]:
rag_run_dir = (
    RESULTS_DIR / "cbd99b81af7e1cb59d122dec8d0cb78717b8d10d/llama-4-maverick-rag2"
)
rag_results_path = (
    rag_run_dir / "healthbench_llama-4-maverick-rag2_20251120_175439_allresults.json"
)
rag_logs_dir = rag_run_dir / "rag_info"
rag_results = AllResults.from_file(rag_results_path)
rag_logs = RAGLog.from_log_dir(rag_logs_dir)

In [6]:
def flatten_metadata(results: AllResults, model_name: str) -> list[dict]:
    """
    Flattens the 'example_level_metadata' objects from the benchmark results file.
    """
    out = []
    for metadata in results.metadata.example_level_metadata:
        out.append(
            {
                "prompt_id": metadata.prompt_id,
                f"score_{model_name}": metadata.score,
            }
        )
    return out


def flatten_rag_logs(rag_logs: list[RAGLog]) -> list[dict]:
    """
    Flattens the rag_logs into dictionaries. It does this by leaving off the
    "conversations" key.
    """
    out = []
    for log in rag_logs:
        out.append(
            {
                "prompt_id": log.prompt_id,
                "vector_search_row_id": log.vector_search_row_id,
                "atropos_case_id": log.atropos_case_id,
                "similarity_score": log.similarity_score,
            }
        )
    return out

In [7]:
baseline = pl.DataFrame(flatten_metadata(baseline_results, "baseline"))
rag_logs_df = pl.DataFrame(flatten_rag_logs(rag_logs))

In [8]:
combined = baseline.join(rag_logs_df, on="prompt_id")
combined.head()

prompt_id,score_baseline,vector_search_row_id,atropos_case_id,similarity_score
str,f64,f64,str,f64
"""5ddbc2ae-a934-4aae-9631-45bac4…",0.583333,9067.0,"""a7cade3239274a8d8cf9718c94f27f…",0.001871
"""9cdbd221-5495-4c79-a10a-f520a5…",-0.101695,7811.0,"""d347b503edc34ee8a6fff5f6a48f66…",0.001998
"""d5972228-1f8d-4f01-9ac0-548c5a…",0.303371,2935.0,"""5650e71ea6db44368a377004ae9a58…",0.002235
"""681cc590-e662-432a-aa0d-b0e8a1…",0.017857,1275.0,"""7716f9896468440a98d5650f136fc6…",0.001632
"""108fa967-dd8f-4d93-905e-751d95…",1.0,43.0,"""2d96c9a70cc44f729510abece9c59c…",0.001594


In [13]:
output_dir = Path("./to_ingest")
eval_input_reader = EvalInputReader()
rag_log_reader = RagLogReader(
    RESULTS_DIR
    / ("cbd99b81af7e1cb59d122dec8d0cb78717b8d10d/llama-4-maverick-rag2/rag_info")
)
baseline_result_metadata_reader = ExampleLevelMetadataReader(baseline_results_path)


def get_axis(tags: list[str]) -> str:
    for tag in tags:
        parts = tag.split(":")
        if parts[0] == "axis":
            return parts[1]
    raise KeyError(tags)


for prompt_id in tqdm(combined["prompt_id"]):
    # Get the input information
    eval_input = eval_input_reader.get_by_prompt_id(prompt_id)
    assert eval_input is not None
    # Get the Atropos Case for this input
    case_id = rag_log_reader.get_case_id_for_prompt_id(prompt_id)
    atropos_summary = get_alexandria_document_by_case_id(case_id)
    assert atropos_summary is not None
    # Write a document for each of the rubric criteria
    for i, rubric_item in enumerate(eval_input.rubrics):
        document = Document(
            question=eval_input.prompt[-1].content,
            criteria=rubric_item.criterion,
            axis=get_axis(rubric_item.tags),
            prompt_id=prompt_id,
            atropos_case_id=case_id,
            atropos_summary=atropos_summary.content,
            criterion_points=rubric_item.points,
        )
        out_file = output_dir / f"{prompt_id}_{i}.json"
        out_file.write_text(document.model_dump_json(indent=2))

  0%|          | 14/5000 [00:14<1:25:22,  1.03s/it]


KeyboardInterrupt: 